This notebook demonstrates how to use the BiomechZoo library to analyze movement data collected from IMU sensors.
The analysis includes data preprocessing, event detection, gait cycle partitioning, normalization, and visualization of the results.

Data source: The data used in this notebook is from the "Losing" dataset, which contains IMU data collected from participants performing
various walking tasks. The data were modified for the purposes of this tutorial.

The full dataset is available at: https://doi.org/10.6084/m9.figshare.c.5758997

The dataset used here is avaialbe in https://github.com/mcgillmotionlab/biomechzoo/tree/main/data/losing/raw

The dataset is described in the following paper: Losing, V., Hasenjäger, M. A Multi-Modal Gait Database of Natural Everyday-Walk in an Urban Environment. Sci Data 9, 473 (2022). https://doi.org/10.1038/s41597-022-01580-3

In [1]:
# import relevant biomech modules
%load_ext autoreload
%autoreload 2
import os
from biomechzoo.biomechzoo import BiomechZoo
from biomechzoo.utils.find_repo_root import find_repo_root

In [2]:
# Step 0: get raw data folder and initialize biomechzoo object
project_root = find_repo_root(os.getcwd())
fld_raw_data = os.path.join(project_root, 'data', 'losing', 'raw')
inplace = False  # tells biomechzoo to create new folders instead of writing to the original folder
verbose = 0 # set verbisity level
name_excludes = ['section_1', 'section_3'] # these trials won't be processed
bmech = BiomechZoo(fld_raw_data, verbose=verbose, inplace=inplace)

In [3]:
# step 1: convert to zoo
# frequency is 60 Hz as per the database documentation
bmech.table2zoo(extension='csv', freq=60, out_folder='1-csv2zoo')

In [4]:
# step 2 clean
ch_remove = ['walk_mode', 'walk_orientation','time_orig']
ch_current = ['jointAngle_jRightKnee_x', 'jointAngle_jRightKnee_y', 'jointAngle_jRightKnee_z']
ch_new = ['RightKneeAngle_x', 'RightKneeAngle_y', 'RightKneeAngle_z']
bmech.removechannel(ch=ch_remove, out_folder='2-clean')
bmech.renamechannel(ch_current=ch_current, ch_new=ch_new, inplace=True)

In [5]:
# inspect the file
bmech.file_info()

File: /Users/phild/Code/biomechzoo/data/losing/2-clean/id16_courseA_50cycles.zoo
Size: 1065552 bytes
Video frequency: 60 Hz
Analog frequency: {} Hz
Channels (40): time, acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z, acceleration_RightLowerLeg_x, acceleration_RightLowerLeg_y, acceleration_RightLowerLeg_z, acceleration_RightFoot_x, acceleration_RightFoot_y, acceleration_RightFoot_z, acceleration_LeftLowerLeg_x, acceleration_LeftLowerLeg_y, acceleration_LeftLowerLeg_z, acceleration_LeftFoot_x, acceleration_LeftFoot_y, acceleration_LeftFoot_z, angularVelocity_Pelvis_x, angularVelocity_Pelvis_y, angularVelocity_Pelvis_z, angularVelocity_RightLowerLeg_x, angularVelocity_RightLowerLeg_y, angularVelocity_RightLowerLeg_z, angularVelocity_RightFoot_x, angularVelocity_RightFoot_y, angularVelocity_RightFoot_z, angularVelocity_LeftLowerLeg_x, angularVelocity_LeftLowerLeg_y, angularVelocity_LeftLowerLeg_z, angularVelocity_LeftFoot_x, angularVelocity_LeftFoot_y, angularVelocity_

In [6]:
# step 3: get foot strikes from Kielmat algorithm
bmech.compute_magnitude(ch_1='acceleration_RightLowerLeg_x', ch_2='acceleration_RightLowerLeg_y', ch_3='acceleration_RightLowerLeg_z', ch_new='acceleration_RightLowerLeg_magnitude', out_folder='3-processing', inplace=False)

bmech.addevent(event_type='mcgrath_fs', event_name='foot_strike', ch='acceleration_RightLowerLeg_z', out_folder='3-processing', inplace=True)

In [7]:
# step 4: partition by gait cycle and normalize
bmech.split_trial_by_gait_cycle(first_event_name='foot_strike_1', out_folder='4-split', inplace=False)
bmech.normalize(inplace=True)

In [9]:
# step 7: plot mean and standard deviation across all files
from biomechzoo.ensembler.ensembler import Ensembler
from biomechzoo.ensembler.plot_spec import PlotSpec
from biomechzoo.ensembler.renderers import MeanSDRenderer
from biomechzoo.ensembler.helpers import ConditionSpec, ConditionSource
from biomechzoo.ensembler.renderers import IndividualLinesRenderer, EventOverlayRenderer, CompositeRenderer
subj_list = ['id14', 'id16', 'id19']

channel = ['RightKneeAngle_z']

spec = ConditionSpec(
    source=ConditionSource.BETWEEN,
    conditions=['courseA', 'courseB', 'courseC'],
)

lines_and_events = CompositeRenderer(IndividualLinesRenderer(), EventOverlayRenderer())     # within stuff

fig = (
    Ensembler(in_folder=bmech.in_folder,  channels=channel, n_rows=1,  n_cols=2, subj_list=subj_list, condition_spec=spec)
    .add_subplot(PlotSpec(channel=channel[0], condition = 'courseA', companions = ['courseB', 'courseC'], row=1, col=1, renderer=IndividualLinesRenderer()))
    .add_subplot(PlotSpec(channel=channel[0], condition = 'courseA', companions = ['courseB', 'courseC'], row=1, col=2, renderer=MeanSDRenderer()))
    .build(title='Joint angles across courses')
)
fig.show()
fig.write_html('walking_analysis.html')